# The Plain Bagel preservation archive

This notebook runs the resumable archiver on Kaggle. Read `README.md` and
`docs/archive-org-setup.md` first.

**Rights:** non-commercial preservation only. Copyright remains with Richard
Coffin / The Plain Bagel. Not public domain, no Creative Commons license, no
commercial license.

**Before bulk upload:** clear the plan with archive.org and the creator. Internet
Archive discourages mirroring still-online material and may remove items or lock
accounts. The current interim collection is Community Video.


In [ ]:
import os
import pathlib
import subprocess
import sys

PROJECT = pathlib.Path(os.environ.get("PLAINBAGEL_PROJECT", "/kaggle/working/arch-yt"))
WORK = pathlib.Path("/kaggle/working/plainbagel-work")
if not (PROJECT / "pyproject.toml").exists():
    raise SystemExit(f"Upload the project to {PROJECT} or set PLAINBAGEL_PROJECT")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT)],
    check=True,
)
print(PROJECT, WORK)

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["IA_ACCESS_KEY"] = secrets.get_secret("IA_ACCESS_KEY")
os.environ["IA_SECRET_KEY"] = secrets.get_secret("IA_SECRET_KEY")
print("Internet Archive secrets loaded")

In [ ]:
def archiver(*args):
    command = [
        sys.executable,
        "-m",
        "plainbagel_archiver",
        "--config",
        str(PROJECT / "config.toml"),
        "--work-dir",
        str(WORK),
        *args,
    ]
    print("$", " ".join(command))
    subprocess.run(command, check=True)


archiver("doctor")

## Discovery

This currently finds the public videos, Shorts, and past livestream VODs. Active
livestreams and upcoming premieres are deferred until they finish.


In [ ]:
archiver("discover")

## First single-item test

This performs real downloads, muxing, and archive.org uploads. Inspect the item
before running a larger batch. Replace the ID if desired.


In [ ]:
archiver("run", "--limit", "1", "--only-id", "OjyPy_4eVdw")
archiver("verify", "--only-id", "OjyPy_4eVdw")

## Batch run

The default time budget is 600 minutes. Kaggle may stop the session earlier; the
next run resumes. Start with a small `--limit` until the item layout and rights
metadata have been reviewed.


In [ ]:
archiver("run", "--limit", "10")
archiver("status")

## Pause, resume, and collection request

Run `pause` to stop after the current video. Run `resume` before continuing.
After 50+ complete items, generate and review the collection request:

```bash
python -m plainbagel_archiver --work-dir /kaggle/working/plainbagel-work collection-request
```

After archive.org creates the dedicated collection, migrate existing items and
then update `collection` in `config.toml` for future uploads.
